In [ ]:
import json
import random
import zipfile
from collections import defaultdict
from google.colab import files

SEED = 42
TRAIN_RATIO = 0.80

datasets = [
    {
        "name": "passport",
        "input": "/content/passport_qa_full.json",
        "train": "passport_qa_train.json",
        "test": "passport_qa_test.json"
    },
    {
        "name": "birth_death",
        "input": "/content/birth_death_qa_full.json",
        "train": "birth_death_qa_train.json",
        "test": "birth_death_qa_test.json"
    }
]

def normalize_text(text):
    return " ".join(text.strip().split())

def grouped_train_test_split(dataset):
    with open(dataset["input"], "r", encoding="utf-8") as f:
        data = json.load(f)

    groups = defaultdict(list)

    # Same output = same paraphrase group
    for item in data:
        key = normalize_text(item.get("output", ""))
        groups[key].append(item)

    group_list = list(groups.values())

    random.seed(SEED)
    random.shuffle(group_list)

    total_items = len(data)
    train_target = int(total_items * TRAIN_RATIO)

    train_data = []
    test_data = []

    for group in group_list:
        if len(train_data) < train_target:
            split_name = "train"
            target = train_data
        else:
            split_name = "test"
            target = test_data

        for item in group:
            item["split"] = split_name
            target.append(item)

    with open(dataset["train"], "w", encoding="utf-8") as f:
        json.dump(train_data, f, ensure_ascii=False, indent=2)

    with open(dataset["test"], "w", encoding="utf-8") as f:
        json.dump(test_data, f, ensure_ascii=False, indent=2)

    print(f"\n{dataset['name']}")
    print("Total:", len(data))
    print("Train:", len(train_data))
    print("Test:", len(test_data))
    print("Groups:", len(group_list))

for dataset in datasets:
    grouped_train_test_split(dataset)

# Zip the 4 files for easy download
zip_name = "train_test_split_files.zip"

with zipfile.ZipFile(zip_name, "w") as zipf:
    for dataset in datasets:
        zipf.write(dataset["train"])
        zipf.write(dataset["test"])

files.download(zip_name)